# Lecture 05 — Web Scraping

PGR215 Data Collection and Analysis — Kristiania University College

## 1. Hva er Web Scraping?

**Web scraping** = automatisk ekstraksjon av data fra nettsider.

Også kalt: screen scraping, web harvesting, web data extraction.

### Bruksområder
- Prissammenligning (e-handel)
- Markedsanalyse og konkurrentovervåking
- Datainnsamling for forskning/ML
- Jobbsøk-aggregering
- Nyhetsovervåking

## 2. Statiske vs. Dynamiske nettsider

| Type | Beskrivelse | Verktøy |
|------|------------|--------|
| **Statisk** | HTML sendes ferdig fra server | `requests` + `BeautifulSoup` |
| **Dynamisk** | JavaScript genererer innhold i nettleseren | `Selenium`, `Playwright` |

### Verktøy-oversikt

| Verktøy | Best for | Kompleksitet |
|---------|---------|-------------|
| **requests + BeautifulSoup** | Statiske sider, enkel scraping | Lav |
| **Selenium** | JS-tunge sider, interaksjon | Medium |
| **Scrapy** | Stor-skala scraping | Høy |
| **Playwright** | Moderne alternativ til Selenium | Medium |

## 3. HTML-grunnlag

For å scrape nettsider må du forstå HTML-struktur:

```html
<html>
  <head><title>Min Side</title></head>
  <body>
    <h1>Overskrift</h1>
    <p class="intro">Tekst her</p>
    <a href="https://example.com">Lenke</a>
    <div id="data">
      <ul>
        <li>Element 1</li>
        <li>Element 2</li>
      </ul>
    </div>
  </body>
</html>
```

### Viktige HTML-konsepter
- **Tags**: `<p>`, `<div>`, `<a>`, `<table>`
- **Attributes**: `class="..."`, `id="..."`, `href="..."`
- **Nesting**: Tags inne i tags (tre-struktur)

In [ ]:
from bs4 import BeautifulSoup

# Parse en enkel HTML-streng
html = """
<html>
<body>
  <h1>Produktliste</h1>
  <div class="products">
    <div class="product">
      <h2>Laptop</h2>
      <span class="price">kr 12.999</span>
      <p class="description">Kraftig bærbar PC</p>
    </div>
    <div class="product">
      <h2>Tastatur</h2>
      <span class="price">kr 899</span>
      <p class="description">Mekanisk tastatur</p>
    </div>
    <div class="product">
      <h2>Mus</h2>
      <span class="price">kr 449</span>
      <p class="description">Trådløs mus</p>
    </div>
  </div>
</body>
</html>
"""

soup = BeautifulSoup(html, 'html.parser')

# Finn elementer
print("=== Grunnleggende BeautifulSoup ===")
print(f"Tittel (h1): {soup.find('h1').text}")
print(f"Første produkt: {soup.find('div', class_='product').find('h2').text}")

# Finn alle produkter
print("\n=== Alle produkter ===")
for prod in soup.find_all('div', class_='product'):
    navn = prod.find('h2').text
    pris = prod.find('span', class_='price').text
    desc = prod.find('p', class_='description').text
    print(f"  {navn}: {pris} — {desc}")

## 4. BeautifulSoup — Viktige metoder

| Metode | Beskrivelse |
|--------|------------|
| `soup.find(tag)` | Finn første element med gitt tag |
| `soup.find(tag, class_='...')` | Finn med CSS-klasse |
| `soup.find(tag, id='...')` | Finn med ID |
| `soup.find_all(tag)` | Finn alle elementer |
| `element.text` | Hent tekst-innholdet |
| `element['href']` | Hent attributt-verdi |
| `element.get('class')` | Hent attributt (returnerer None hvis mangler) |
| `soup.select('css selector')` | Bruk CSS-selektorer |

In [ ]:
# CSS-selektorer i BeautifulSoup
print("=== CSS Selektorer ===")

# Alle h2 i .product
produkter = soup.select('.product h2')
print(f"Produktnavn: {[p.text for p in produkter]}")

# Alle priser
priser = soup.select('.price')
print(f"Priser: {[p.text for p in priser]}")

# Navigere DOM-treet
print("\n=== Navigering ===")
first_product = soup.find('div', class_='product')
print(f"Parent: {first_product.parent.get('class')}")
print(f"Children: {[child.name for child in first_product.children if child.name]}")

## 5. Scrape en HTML-tabell

In [ ]:
# Parse en HTML-tabell
import pandas as pd

html_table = """
<table id="students">
  <thead>
    <tr><th>Navn</th><th>Alder</th><th>Karakter</th></tr>
  </thead>
  <tbody>
    <tr><td>Anna</td><td>22</td><td>A</td></tr>
    <tr><td>Erik</td><td>25</td><td>B</td></tr>
    <tr><td>Sara</td><td>23</td><td>A</td></tr>
    <tr><td>Ole</td><td>24</td><td>C</td></tr>
  </tbody>
</table>
"""

soup = BeautifulSoup(html_table, 'html.parser')
table = soup.find('table', id='students')

# Manuell parsing
headers = [th.text for th in table.find('thead').find_all('th')]
rows = []
for tr in table.find('tbody').find_all('tr'):
    row = [td.text for td in tr.find_all('td')]
    rows.append(row)

df = pd.DataFrame(rows, columns=headers)
print("Manuelt parset tabell:")
display(df)

# Enklere: pd.read_html()
print("\nMed pd.read_html():")
dfs = pd.read_html(html_table)
display(dfs[0])

## 6. Requests — Hente nettsider

In [ ]:
import requests

# Hent en nettside (eksempel med httpbin for testing)
try:
    response = requests.get('https://httpbin.org/html', timeout=5)
    print(f"Status code: {response.status_code}")
    print(f"Content type: {response.headers.get('content-type')}")
    print(f"Lengde: {len(response.text)} tegn")
    
    # Parse med BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')
    print(f"\nTittel: {soup.find('h1').text if soup.find('h1') else 'Ingen h1'}")
    
except requests.exceptions.RequestException as e:
    print(f"Kunne ikke koble til: {e}")
    print("(Krever internett-tilgang)")

## 7. Komplett scraping-eksempel

Eksempel: Scrape en lokal HTML-fil med produktdata.

In [ ]:
# Lag en lokal HTML-fil å scrape
html_page = """
<!DOCTYPE html>
<html>
<head><title>Nettbutikk</title></head>
<body>
<h1>Elektronikk</h1>
<div class="product-list">
  <div class="product" data-id="1">
    <h3><a href="/product/1">Laptop Pro 15</a></h3>
    <span class="price">kr 15.999,-</span>
    <span class="rating">4.5/5</span>
    <span class="stock in-stock">På lager</span>
  </div>
  <div class="product" data-id="2">
    <h3><a href="/product/2">Mekanisk Tastatur K70</a></h3>
    <span class="price">kr 1.299,-</span>
    <span class="rating">4.8/5</span>
    <span class="stock out-of-stock">Utsolgt</span>
  </div>
  <div class="product" data-id="3">
    <h3><a href="/product/3">Gaming Mus G502</a></h3>
    <span class="price">kr 699,-</span>
    <span class="rating">4.6/5</span>
    <span class="stock in-stock">På lager</span>
  </div>
  <div class="product" data-id="4">
    <h3><a href="/product/4">4K Monitor 27"</a></h3>
    <span class="price">kr 4.499,-</span>
    <span class="rating">4.3/5</span>
    <span class="stock in-stock">På lager</span>
  </div>
</div>
</body>
</html>
"""

import re

soup = BeautifulSoup(html_page, 'html.parser')

produkter = []
for prod in soup.find_all('div', class_='product'):
    navn = prod.find('h3').text.strip()
    link = prod.find('a')['href']
    pris_tekst = prod.find('span', class_='price').text
    pris = float(re.sub(r'[^\d]', '', pris_tekst))  # Trekk ut tall
    rating = float(prod.find('span', class_='rating').text.split('/')[0])
    stock = 'in-stock' in prod.find('span', class_='stock').get('class', [])
    
    produkter.append({
        'navn': navn,
        'link': link,
        'pris_kr': pris,
        'rating': rating,
        'pa_lager': stock
    })

df_products = pd.DataFrame(produkter)
print("Scraped produktdata:")
display(df_products)

# Analyse
print(f"\nGjennomsnittspris: kr {df_products['pris_kr'].mean():,.0f}")
print(f"Høyest ratede: {df_products.loc[df_products['rating'].idxmax(), 'navn']}")
print(f"På lager: {df_products['pa_lager'].sum()} av {len(df_products)}")

## 8. Etikk og juss

### robots.txt
Før du scraper, sjekk `robots.txt` på nettstedet:
```
https://example.com/robots.txt
```

### Viktige regler:
1. **Respekter robots.txt** — noen sider forbyr scraping
2. **Rate limiting** — ikke overbelast serveren (`time.sleep()`)
3. **Terms of Service** — les vilkårene
4. **Personvern** — ikke scrape persondata uten grunn
5. **User-Agent** — identifiser deg selv i requests
6. **Bruk API først** — hvis tilgjengelig, foretrekk API over scraping

In [ ]:
# Sjekk robots.txt
try:
    resp = requests.get('https://www.vg.no/robots.txt', timeout=5)
    print("=== robots.txt for vg.no ===")
    print(resp.text[:500])
except:
    print("Kunne ikke hente robots.txt (krever internett)")
    print("\nTypisk robots.txt-format:")
    print("User-agent: *")
    print("Disallow: /admin/")
    print("Disallow: /private/")
    print("Allow: /")
    print("Crawl-delay: 10")

## 9. Selenium — Dynamiske sider

For JavaScript-tunge sider brukes Selenium:

```python
from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()
driver.get("https://example.com")

# Vent på at JS laster
element = driver.find_element(By.CLASS_NAME, "product")
print(element.text)

driver.quit()
```

### Når bruke Selenium?
- Sider som bruker React, Angular, Vue
- Sider som krever login
- Sider med infinite scroll
- Sider som laster data med AJAX

In [ ]:
# Selenium-eksempel (pseudo-kode, krever Chrome + chromedriver)
print("=== Selenium vs. BeautifulSoup ===")
print()
print("BeautifulSoup:")
print("  + Rask og enkel")
print("  + Lite ressursbruk")
print("  - Kun statisk HTML")
print("  - Ingen JS-rendering")
print()
print("Selenium:")
print("  + Rendrer JavaScript")
print("  + Kan interagere (klikke, skrive)")
print("  + Håndterer login, scroll, etc.")
print("  - Treg (åpner faktisk nettleser)")
print("  - Mer kompleks oppsett")
print()
print("Scrapy:")
print("  + Bygget for stor-skala scraping")
print("  + Innebygd håndtering av links, paginering")
print("  + Asynkron (veldig rask)")
print("  - Brattere læringskurve")

## Oppsummering

**Nøkkelkonsepter fra Lecture 05:**

1. Web scraping = automatisk dataekstraksjon fra nettsider
2. HTML-struktur: tags, attributes, nesting
3. `requests` for å hente HTML
4. `BeautifulSoup` for å parse og navigere HTML
5. `.find()`, `.find_all()`, `.select()` for å finne elementer
6. `.text` for innhold, `['href']` for attributter
7. `pd.read_html()` for tabeller
8. Selenium for dynamiske (JS) sider
9. Respekter robots.txt og bruk rate limiting